## Building an Expected Goals (xG) Model

### 1. Data Collection
**StatsBomb open data** containing event and match information.
- One competition to test the logic: World Cup 2018
- Scale up to all competitions

---

### 2. Data Preparation
Get familiar with the data. 

---

### 3. Feature Engineering
Create meaningful variables that influence goal probability, such as **shot distance** and **angle**, as well as dummy variables where needed.

---

### 4. Model Development
Choose, train and compare a statistical or machine learning model to predict the probability of a goal from shot features.
- **logistic regression**
- **XGBoost**

---

### 5. Evaluation
Compare predictions to real outcomes or benchmark xG models. Assess model performance different metrics. Calculate xG of a competition/team/player to gain insights and visualize results, for example **shot maps**.

---

### 6. Deployment
Integrate the model into an application.

# 1. Data Collection

In [1]:
#pip install statsbombpy

In [2]:
#pip install mplsoccer

In [3]:
#pip install xgboost

In [4]:
#pip install pyarrow

In [7]:
#pip install joblib

In [9]:
#pip install streamlit

In [11]:
import warnings
from statsbombpy import sb
warnings.filterwarnings("ignore",message="credentials were not supplied. open data access only")

In [12]:
#Load competitions
competitions = sb.competitions()
competitions

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,None,None,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2024-09-28T01:57:35.846538,None,None,2024-09-28T01:57:35.846538
3,16,4,Europe,Champions League,male,False,False,2018/2019,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,None,2025-05-08T15:10:50.835274
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
70,35,75,Europe,UEFA Europa League,male,False,False,1988/1989,2024-02-12T14:45:05.702250,2021-06-13T16:17:31.694,None,2024-02-12T14:45:05.702250
71,53,315,Europe,UEFA Women's Euro,female,False,True,2025,2025-07-28T14:19:20.467348,2025-07-29T16:03:07.355174,2025-07-29T16:03:07.355174,2025-07-28T14:19:20.467348
72,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2024-02-13T13:27:17.178263,2024-02-13T13:30:52.820588,2024-02-13T13:30:52.820588,2024-02-13T13:27:17.178263
73,72,107,International,Women's World Cup,female,False,True,2023,2025-07-14T10:07:06.620906,2025-07-14T10:10:27.224586,2025-07-14T10:10:27.224586,2025-07-14T10:07:06.620906


In [13]:
#Load matches from competition: 2018 World Cup
matches = sb.matches(competition_id=43, season_id=3)
matches

,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,...,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
0,7585,2018-07-03,20:00:00.000,International - FIFA World Cup,2018,Colombia,England,1,1,available,...,2021-06-13T16:17:31.694,4,Round of 16,Otkritie Bank Arena,Mark Geiger,José Néstor Pekerman,Gareth Southgate,1.0.2,None,None
1,7570,2018-06-28,20:00:00.000,International - FIFA World Cup,2018,England,Belgium,0,1,available,...,2021-06-13T16:17:31.694,3,Group Stage,Stadion Kaliningrad,Damir Skomina,Gareth Southgate,Roberto Martínez Montoliú,1.0.2,None,None
2,7586,2018-07-03,16:00:00.000,International - FIFA World Cup,2018,Sweden,Switzerland,1,0,available,...,2021-06-13T16:17:31.694,4,Round of 16,Saint-Petersburg Stadium,Damir Skomina,Jan Olof Andersson,Vladimir Petković,1.0.2,None,None
3,7557,2018-06-25,20:00:00.000,International - FIFA World Cup,2018,Iran,Portugal,1,1,available,...,2021-06-13T16:17:31.694,3,Group Stage,Mordovia Arena,Enrique Cáceres,Carlos Manuel Brito Leal Queiróz,Fernando Manuel Fernandes da Costa Santos,1.0.2,None,None
4,7542,2018-06-20,14:00:00.000,International - FIFA World Cup,2018,Portugal,Morocco,1,0,available,...,2021-06-13T16:17:31.694,2,Group Stage,Stadion Luzhniki,Mark Geiger,Fernando Manuel Fernandes da Costa Santos,Hervé Renard,1.0.2,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,7540,2018-06-19,20:00:00.000,International - FIFA World Cup,2018,Russia,Egypt,3,1,available,...,2021-06-13T16:17:31.694,2,Group Stage,Saint-Petersburg Stadium,Enrique Cáceres,Stanislav Cherchesov,Héctor Raúl Cúper,1.0.2,None,None
60,8652,2018-07-07,20:00:00.000,International - FIFA World Cup,2018,Russia,Croatia,2,2,available,...,2021-06-13T16:17:31.694,5,Quarter-finals,\tOlimpiyskiy Stadion Fisht,Sandro Ricci,Stanislav Cherchesov,Zlatko Dalić,1.0.2,None,None
61,7563,2018-06-26,16:00:00.000,International - FIFA World Cup,2018,Denmark,France,0,0,available,...,2021-06-13T16:17:31.694,3,Group Stage,Stadion Luzhniki,Sandro Ricci,Åge Fridtjof Hareide,Didier Deschamps,1.0.2,None,None
62,7556,2018-06-24,17:00:00.000,International - FIFA World Cup,2018,Japan,Senegal,2,2,available,...,2021-06-13T16:17:31.694,2,Group Stage,\tEkaterinburg Arena,Gianluca Rocchi,Akira Nishino,Aliou Cissé,1.0.2,None,None


In [14]:
events = sb.events(matches["match_id"].iloc[0])
events

,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,block_offensive,carry_end_location,clearance_aerial_won,counterpress,dribble_nutmeg,...,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 433, 'lineup': [{'player': {'id'...",Colombia,769,00:00:00.000,Starting XI,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 352, 'lineup': [{'player': {'id'...",England,768,00:00:00.000,Starting XI,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,England,768,00:00:00.000,Half Start,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Colombia,769,00:00:00.000,Half Start,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Colombia,769,00:00:00.000,Half Start,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Tactical,103.0,Marcus Rashford,3318.0,NaN,England,768,00:07:33.493,Substitution,NaN
4015,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Tactical,103.0,Cristian Eduardo Zapata Valencia,6360.0,NaN,Colombia,769,00:10:17.880,Substitution,NaN
4016,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 442, 'lineup': [{'player': {'id'...",Colombia,769,00:16:06.374,Tactical Shift,NaN
4017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Colombia,769,00:31:38.320,Camera off,NaN


In [15]:
import pandas as pd

all_events = []

# Loop through each match
for match_id in matches["match_id"]:
    events = sb.events(match_id=match_id)
    events["match_id"] = match_id
    all_events.append(events)
# Concatenate all event DataFrames into one
events_df = pd.concat(all_events, ignore_index=True)
print("✅ All match events collected and saved!")
events_df

✅ All match events collected and saved!


,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,block_offensive,carry_end_location,clearance_aerial_won,counterpress,dribble_nutmeg,...,foul_committed_offensive,injury_stoppage_in_chain,pass_cut_back,pass_technique,pass_through_ball,shot_one_on_one,ball_recovery_offensive,miscontrol_aerial_won,pass_miscommunication,shot_redirect
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227844,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
227845,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
227846,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
227847,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 2. Data Preparation

In [16]:
events_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 227849 entries, 0 to 227848
Data columns (total 95 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   50_50                           108 non-null     object 
 1   bad_behaviour_card              40 non-null      object 
 2   ball_receipt_outcome            8800 non-null    object 
 3   ball_recovery_recovery_failure  512 non-null     object 
 4   block_deflection                79 non-null      object 
 5   block_offensive                 43 non-null      object 
 6   carry_end_location              50521 non-null   object 
 7   clearance_aerial_won            151 non-null     object 
 8   counterpress                    6470 non-null    object 
 9   dribble_nutmeg                  99 non-null      object 
 10  dribble_outcome                 2109 non-null    object 
 11  duel_outcome                    1830 non-null    object 
 12  duel_type       

In [17]:
for column in events_df:
    if column in ["tactics","pass_recipient","pass_recipient_id","player","player_id",
                  "substitution_replacement","substitution_replacement_id"]:
        continue
    if events_df[column].apply(lambda x: isinstance(x, (dict,list))).any():
        events_df[column] = events_df[column].apply(str)

    print(f"\nColumn: {column}")
    print(events_df[column].unique())


Column: 50_50
['nan' "{'outcome': {'id': 1, 'name': 'Lost'}}"
 "{'outcome': {'id': 3, 'name': 'Success To Team'}}"
 "{'outcome': {'id': 2, 'name': 'Success To Opposition'}}"
 "{'outcome': {'id': 4, 'name': 'Won'}}"]

Column: bad_behaviour_card
[nan 'Yellow Card']

Column: ball_receipt_outcome
[nan 'Incomplete']

Column: ball_recovery_recovery_failure
[nan True]

Column: block_deflection
[nan True]

Column: block_offensive
[nan True]

Column: carry_end_location
['nan' '[51.0, 40.0]' '[49.0, 55.0]' ... '[90.0, 46.0]' '[15.0, 23.0]'
 '[103.0, 38.0]']

Column: clearance_aerial_won
[nan True]

Column: counterpress
[nan True]

Column: dribble_nutmeg
[nan True]

Column: dribble_outcome
[nan 'Incomplete' 'Complete']

Column: duel_outcome
[nan 'Lost In Play' 'Success In Play' 'Won' 'Lost Out' 'Success Out']

Column: duel_type
[nan 'Tackle' 'Aerial Lost']

Column: duration
[0.    0.754 9.32  ... 4.268 3.407 6.679]

Column: foul_committed_advantage
[nan True]

Column: foul_committed_card
[nan 'Y


Column: shot_deflected
[nan True]

Column: shot_end_location
['nan' '[120.0, 42.2, 2.0]' '[113.0, 53.0]' ... '[120.0, 44.2, 4.2]'
 '[104.0, 32.0]' '[119.0, 36.8, 0.8]']

Column: shot_first_time
[nan True]

Column: shot_follows_dribble
[nan True]

Column: shot_freeze_frame
['nan'
 "[{'location': [116.0, 41.0], 'player': {'id': 10955, 'name': 'Harry Kane'}, 'position': {'id': 24, 'name': 'Left Center Forward'}, 'teammate': True}, {'location': [95.0, 42.0], 'player': {'id': 3308, 'name': 'Kieran Trippier'}, 'position': {'id': 12, 'name': 'Right Midfield'}, 'teammate': True}, {'location': [109.0, 39.0], 'player': {'id': 3532, 'name': 'Jordan Brian Henderson'}, 'position': {'id': 10, 'name': 'Center Defensive Midfield'}, 'teammate': True}, {'location': [109.0, 49.0], 'player': {'id': 3336, 'name': 'Harry Maguire'}, 'position': {'id': 5, 'name': 'Left Center Back'}, 'teammate': True}, {'location': [110.0, 47.0], 'player': {'id': 3244, 'name': 'John Stones'}, 'position': {'id': 4, 'name': 'C


Column: shot_statsbomb_xg
[       nan 0.00981611 0.03820353 ... 0.08890756 0.02032534 0.02102943]

Column: shot_technique
[nan 'Normal' 'Volley' 'Diving Header' 'Half Volley' 'Lob' 'Overhead Kick'
 'Backheel']

Column: shot_type
[nan 'Free Kick' 'Open Play' 'Penalty']

Column: substitution_outcome
[nan 'Tactical' 'Injury']

Column: substitution_outcome_id
[ nan 103. 102.]

Column: team
['Colombia' 'England' 'Belgium' 'Sweden' 'Switzerland' 'Iran' 'Portugal'
 'Morocco' 'Croatia' 'Panama' 'Serbia' 'Brazil' 'Costa Rica' 'Egypt'
 'Uruguay' 'South Korea' 'Mexico' 'Saudi Arabia' 'Poland' 'France' 'Peru'
 'Senegal' 'Spain' 'Germany' 'Tunisia' 'Argentina' 'Denmark' 'Nigeria'
 'Iceland' 'Australia' 'Russia' 'Japan']

Column: team_id
[769 768 782 790 773 797 780 788 785 798 786 781 795 774 783 791 794 799
 789 771 784 787 772 770 777 779 776 775 793 792 796 778]

Column: timestamp
['00:00:00.000' '00:00:00.240' '00:00:02.120' ... '00:35:26.800'
 '00:51:37.840' '00:50:33.160']

Column: type
['St

### Columns to consider:
- goalkeeper_end_location
- goalkeeper_position
- goalkeeper_technique
- location
- period
- position
- shot_body_part
- shot_first_time
- shot_technique
- under_pressure
- shot_one_on_one
<br><br>
- **shot_type** (Open Play)
- **shot_outcome** (Goal)
- **shot_statsbomb_xg**

In [18]:
events_df["under_pressure"] = events_df.apply(lambda row: 1 if row["under_pressure"]==True else 0, axis=1)
events_df["under_pressure"].unique()

array([0, 1], dtype=int64)

In [19]:
events_df["goal"] = events_df.apply(lambda row:1 if row['shot_outcome']=='Goal' else 0, axis=1)
events_df["goal"].unique()

array([0, 1], dtype=int64)

In [20]:
events_df["shot_first_time"] = events_df.apply(lambda row:1 if row['shot_first_time']==True else 0, axis=1)
events_df["shot_first_time"].unique()

array([0, 1], dtype=int64)

In [21]:
events_df["shot_one_on_one"] = events_df.apply(lambda row:1 if row['shot_one_on_one']==True else 0, axis=1)
events_df["shot_one_on_one"].unique()

array([0, 1], dtype=int64)

In [22]:
round(events_df.groupby("shot_type")[["goal"]].sum()/events_df.groupby("shot_type")[["goal"]].count()*100,2).applymap(lambda x: f"{x}%")

,goal
shot_type,
Free Kick,7.32%
Open Play,8.29%
Penalty,70.59%


In [23]:
round(events_df.groupby("shot_body_part")[["goal"]].sum()/events_df.groupby("shot_body_part")[["goal"]].count()*100,2).applymap(lambda x: f"{x}%")

,goal
shot_body_part,
Head,10.32%
Left Foot,9.4%
Other,7.14%
Right Foot,11.68%


In [24]:
round(events_df.groupby("shot_technique")[["goal"]].sum()/events_df.groupby("shot_technique")[["goal"]].count()*100,2).applymap(lambda x: f"{x}%")

,goal
shot_technique,
Backheel,16.67%
Diving Header,8.33%
Half Volley,7.63%
Lob,45.45%
Normal,10.99%
Overhead Kick,0.0%
Volley,8.55%


**Consider only Open Play because Penalties have much higher chance of scoring.**

In [25]:
sample = events_df[events_df["shot_type"]=="Open Play"].reset_index()

In [26]:
sample[['x', 'y']] = (sample['location'].str.strip('[]').str.split(',', expand=True).astype(float))
sample[['x','y']]

,x,y
0,112.0,54.0
1,98.0,37.0
2,119.0,36.0
3,97.0,56.0
4,90.0,18.0
...,...,...
1551,98.0,51.0
1552,102.0,22.0
1553,116.0,30.0
1554,101.0,58.0


In [27]:
from mplsoccer import VerticalPitch

# visualizing shots

# filter goals / non-shot goals
df_goals = sample[sample["shot_outcome"] == 'Goal'].copy()
df_non_goal_shots = sample[sample["shot_outcome"] != 'Goal'].copy()

# setup the pitch
pitch = VerticalPitch(pad_bottom=0.5,  # pitch extends slightly below halfway line
                      half=True,  # half of a pitch
                      goal_type='box',
                      goal_alpha=0.8, pitch_color='#22312b', line_color='#c7d5cc')  # control the goal transparency

fig, ax = pitch.draw(figsize=(8, 6))

sc1 = pitch.scatter(df_non_goal_shots["x"], df_non_goal_shots["y"],
                    c='#ba4f45',
                    marker='o',
                    ax=ax)
sc2 = pitch.scatter(df_goals["x"], df_goals["y"],
                    c='#ad993c',
                    marker='o',
                    ax=ax)

# 3. Feature Engineering

In [28]:
import numpy as np
import math

def angle(x, y):
  # 44 and 36 is the location of each goal post
  g0 = [120, 44]
  p = [x, y]
  g1 = [120, 36]

  v0 = np.array(g0) - np.array(p)
  v1 = np.array(g1) - np.array(p)

  angle = np.math.atan2(np.linalg.det([v0,v1]),np.dot(v0,v1))
  return(abs(np.degrees(angle)))

def distance(x, y):
    # center of goal midpoint between 36 and 44
    x_dist = 120 - x
    y_dist = 40 - y
    
    return math.sqrt(x_dist**2 + y_dist**2)

In [29]:
sample["angle"] = sample.apply(lambda row: angle(row["x"],row["y"]), axis=1)
sample["distance"] = sample.apply(lambda row: distance(row["x"],row["y"]), axis=1)

In [30]:
train_sample = sample[["angle","distance","period","under_pressure","shot_first_time","shot_one_on_one","position",
                       "shot_body_part","shot_technique","shot_statsbomb_xg","goal"]]
train_sample_X = train_sample[["angle","distance","period","under_pressure","shot_first_time","shot_one_on_one","position",
                               "shot_body_part","shot_technique"]]
train_sample_y = train_sample[["goal"]]

In [31]:
train_sample_X = pd.get_dummies(train_sample_X, columns = ["shot_body_part"])
train_sample_X = pd.get_dummies(train_sample_X, columns = ["shot_technique"])
train_sample_X = pd.get_dummies(train_sample_X, columns = ["position"])
train_sample_X = pd.get_dummies(train_sample_X, columns = ["period"])
train_sample_X

,angle,distance,under_pressure,shot_first_time,shot_one_on_one,shot_body_part_Head,shot_body_part_Left Foot,shot_body_part_Other,shot_body_part_Right Foot,shot_technique_Backheel,...,position_Right Center Midfield,position_Right Defensive Midfield,position_Right Midfield,position_Right Wing,position_Right Wing Back,position_Secondary Striker,period_1,period_2,period_3,period_4
0,14.697319,16.124515,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
1,20.252686,22.203603,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
2,82.874984,4.123106,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,13.456275,28.017851,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
4,9.950627,37.202150,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1551,16.636753,24.596748,0,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,1,0,0
1552,12.835609,25.455844,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
1553,17.744672,10.770330,0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
1554,12.800564,26.172505,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0


In [32]:
train_sample = pd.get_dummies(train_sample, columns = ["shot_body_part"])
train_sample = pd.get_dummies(train_sample, columns = ["shot_technique"])
train_sample = pd.get_dummies(train_sample, columns = ["position"])
train_sample = pd.get_dummies(train_sample, columns = ["period"])
train_sample.groupby(["goal"]).mean().T

goal,0,1
angle,24.291690,42.781434
distance,19.434957,12.282136
under_pressure,0.217239,0.147287
shot_first_time,0.220743,0.325581
shot_one_on_one,0.027330,0.124031
shot_statsbomb_xg,0.079865,0.252734
shot_body_part_Head,0.194814,0.248062
shot_body_part_Left Foot,0.299229,0.294574
shot_body_part_Other,0.009110,0.007752
shot_body_part_Right Foot,0.496847,0.449612


In [33]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(30, 20))
sns.heatmap(train_sample_X.corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.show()

# 4. Model Development

In [34]:
from sklearn.preprocessing import StandardScaler
numeric_cols = ["angle", "distance"]
scaler = StandardScaler()
train_sample_X[numeric_cols] = scaler.fit_transform(train_sample_X[numeric_cols])

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
LogRegModel = LogisticRegression()
LogRegModel.fit(train_sample_X, train_sample_y)
y_pred = LogRegModel.predict_proba(train_sample_X)[:, 1]
metrics.r2_score(train_sample_y, y_pred)

0.15578823363380645

Standardization R2: 0.15578823363380645

No standardization R2: 0.15598483135318297

Previous R2: 0.12877920747510807

In [36]:
metrics.r2_score(train_sample_y, train_sample["shot_statsbomb_xg"])

0.16097825474300864

In [37]:
sample_with_xg = train_sample_X.copy()
sample_with_xg["xG_pred"] = y_pred

In [38]:
corr = sample_with_xg[[col for col in sample_with_xg.columns if col != "xG_pred"]].corrwith(sample_with_xg["xG_pred"])
corr_df = corr.to_frame(name='xG').T
plt.figure(figsize=(30, 0.7))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap='coolwarm')
plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold')
plt.show()

In [39]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    objective='binary:logistic',  # output = probability of class 1
    eval_metric='logloss',        # common for probabilistic models
    use_label_encoder=False,
    n_estimators=200,             # number of trees
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(train_sample_X, train_sample_y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [40]:
y_pred_xgb = xgb_model.predict_proba(train_sample_X)[:, 1]
print("R²:", metrics.r2_score(train_sample_y, y_pred_xgb))

R²: 0.3529688322825669


In [41]:
import matplotlib.pyplot as plt
from xgboost import plot_importance

plot_importance(xgb_model, max_num_features=10)
plt.show()

In [42]:
sample_with_xg = train_sample_X.copy()
sample_with_xg["xG_pred"] = y_pred_xgb

### Scale up: Load other competitions

In [43]:
competitions[(competitions["competition_id"].isin([43,55,35,16,2,11,7])) & (competitions["season_name"]>="2010")]

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
3,16,4,Europe,Champions League,male,False,False,2018/2019,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,None,2025-05-08T15:10:50.835274
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,None,2024-02-13T02:35:28.134882
5,16,2,Europe,Champions League,male,False,False,2016/2017,2024-02-13T02:37:32.205154,2021-06-13T16:17:31.694,None,2024-02-13T02:37:32.205154
6,16,27,Europe,Champions League,male,False,False,2015/2016,2024-06-12T07:45:38.786894,2021-06-13T16:17:31.694,None,2024-06-12T07:45:38.786894
7,16,26,Europe,Champions League,male,False,False,2014/2015,2024-02-12T12:49:54.914228,2021-06-13T16:17:31.694,None,2024-02-12T12:49:54.914228
8,16,25,Europe,Champions League,male,False,False,2013/2014,2024-02-12T12:48:48.479157,2021-06-13T16:17:31.694,None,2024-02-12T12:48:48.479157
9,16,24,Europe,Champions League,male,False,False,2012/2013,2024-02-12T12:47:34.340413,2021-06-13T16:17:31.694,None,2024-02-12T12:47:34.340413
10,16,23,Europe,Champions League,male,False,False,2011/2012,2024-02-13T02:36:35.698340,2021-06-13T16:17:31.694,None,2024-02-13T02:36:35.698340
11,16,22,Europe,Champions League,male,False,False,2010/2011,2024-02-12T12:53:03.944320,2021-06-13T16:17:31.694,None,2024-02-12T12:53:03.944320
29,43,106,International,FIFA World Cup,male,False,True,2022,2024-12-16T10:15:11.055845,2024-12-16T10:21:13.710934,2024-12-16T10:21:13.710934,2024-12-16T10:15:11.055845


In [44]:
events_all = pd.read_parquet("events_all.parquet")
events_all

,location,period,under_pressure,shot_first_time,shot_one_on_one,position,shot_body_part,shot_technique,shot_statsbomb_xg,shot_outcome,shot_type,team,player,match_id,competition_id,season_id
0,None,1,None,None,None,None,None,None,NaN,None,None,Bayer Leverkusen,None,3895302,9,281
1,None,1,None,None,None,None,None,None,NaN,None,None,Werder Bremen,None,3895302,9,281
2,None,1,None,None,None,None,None,None,NaN,None,None,Bayer Leverkusen,None,3895302,9,281
3,None,1,None,None,None,None,None,None,NaN,None,None,Werder Bremen,None,3895302,9,281
4,None,2,None,None,None,None,None,None,NaN,None,None,Bayer Leverkusen,None,3895302,9,281
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8790985,None,2,None,None,None,Left Defensive Midfield,None,None,NaN,None,None,Japan,Gaku Shibasaki,7584,43,3
8790986,None,2,None,None,None,Right Wing,None,None,NaN,None,None,Japan,Genki Haraguchi,7584,43,3
8790987,"[117.0, 16.0]",2,True,None,None,Right Center Midfield,None,None,NaN,None,None,Belgium,Kevin De Bruyne,7584,43,3
8790988,"[113.0, 41.0]",2,None,None,None,Right Attacking Midfield,None,None,NaN,None,None,Belgium,Marouane Fellaini-Bakkioui,7584,43,3


In [45]:
events_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790990 entries, 0 to 8790989
Data columns (total 16 columns):
 #   Column             Dtype  
---  ------             -----  
 0   location           object 
 1   period             int64  
 2   under_pressure     object 
 3   shot_first_time    object 
 4   shot_one_on_one    object 
 5   position           object 
 6   shot_body_part     object 
 7   shot_technique     object 
 8   shot_statsbomb_xg  float64
 9   shot_outcome       object 
 10  shot_type          object 
 11  team               object 
 12  player             object 
 13  match_id           int64  
 14  competition_id     int64  
 15  season_id          int64  
dtypes: float64(1), int64(4), object(11)
memory usage: 1.0+ GB


In [46]:
from sklearn.model_selection import train_test_split
events_all = events_all[events_all["location"].notna()]
sample = events_all[events_all["shot_type"]=="Open Play"].reset_index()
sample["goal"] = sample.apply(lambda row:1 if row['shot_outcome']=='Goal' else 0, axis=1)
sample["under_pressure"] = sample.apply(lambda row: 1 if row["under_pressure"]==True else 0, axis=1)
sample["shot_first_time"] = sample.apply(lambda row:1 if row['shot_first_time']==True else 0, axis=1)
sample["shot_one_on_one"] = sample.apply(lambda row:1 if row['shot_one_on_one']==True else 0, axis=1)
sample[['x', 'y']] = pd.DataFrame(sample['location'].tolist(), index=sample.index)
sample["angle"] = sample.apply(lambda row: angle(row["x"],row["y"]), axis=1)
sample["distance"] = sample.apply(lambda row: distance(row["x"],row["y"]), axis=1)
train_sample = sample[["angle","distance","period","under_pressure","shot_first_time","shot_one_on_one","position",
                       "shot_body_part","shot_technique","shot_statsbomb_xg","goal"]]
X = train_sample.drop(["shot_statsbomb_xg","goal"], axis=1)
y = train_sample["goal"]
categorical_cols = ["shot_body_part", "shot_technique", "position", "period"]
X = pd.get_dummies(X, columns=categorical_cols)
train_sample = pd.get_dummies(train_sample, columns=categorical_cols)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [47]:
train_sample.groupby(["goal"]).mean().T

goal,0,1
angle,23.931080,39.378069
distance,19.520393,12.586994
under_pressure,0.266356,0.236376
shot_first_time,0.318226,0.449300
shot_one_on_one,0.046192,0.131095
shot_statsbomb_xg,0.079137,0.265531
shot_body_part_Head,0.168334,0.173275
shot_body_part_Left Foot,0.323401,0.316349
shot_body_part_Other,0.002433,0.008267
shot_body_part_Right Foot,0.505832,0.502109


In [48]:
plt.figure(figsize=(30, 20))
sns.heatmap(X_train.corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.show()

In [49]:
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
LogRegModel = LogisticRegression()
LogRegModel.fit(X_train, y_train)
y_pred = LogRegModel.predict_proba(X_test)[:, 1]
metrics.r2_score(y_test, y_pred)

0.15340847416536008

In [50]:
metrics.r2_score(y_test, X_test.join(train_sample["shot_statsbomb_xg"])["shot_statsbomb_xg"])

0.20811368087993742

In [51]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss
print("ROC-AUC:", roc_auc_score(y_test, y_pred))
print("PR-AUC:", average_precision_score(y_test, y_pred))
print("Log Loss:", log_loss(y_test, y_pred))
print("Brier Score:", brier_score_loss(y_test, y_pred))
print("xG Sum / Goals Ratio:", y_pred.sum() / y_test.sum())

ROC-AUC: 0.7937066520669729
PR-AUC: 0.37548861370483266
Log Loss: 0.27227344068116616
Brier Score: 0.0779885386661362
xG Sum / Goals Ratio: 1.0025027178487276


In [52]:
sample_with_xg = X_test.copy()
sample_with_xg["xG_pred"] = y_pred

In [53]:
corr = sample_with_xg[[col for col in sample_with_xg.columns if col != "xG_pred"]].corrwith(sample_with_xg["xG_pred"])
corr_df = corr.to_frame(name='xG').T
plt.figure(figsize=(30, 0.7))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap='coolwarm')
plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold')
plt.show()

In [54]:
xgb_model = XGBClassifier(
    objective='binary:logistic',  # output = probability of class 1
    eval_metric='logloss',        # common for probabilistic models
    use_label_encoder=False,
    n_estimators=200,             # number of trees
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [55]:
y_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]
print("R²:", metrics.r2_score(y_test, y_pred_xgb))

R²: 0.15708439233170923


In [56]:
print("ROC-AUC:", roc_auc_score(y_test, y_pred_xgb))
print("PR-AUC:", average_precision_score(y_test, y_pred_xgb))
print("Log Loss:", log_loss(y_test, y_pred_xgb))
print("Brier Score:", brier_score_loss(y_test, y_pred_xgb))
print("xG Sum / Goals Ratio:", y_pred_xgb.sum() / y_test.sum())

ROC-AUC: 0.7957407918867415
PR-AUC: 0.3769376825192118
Log Loss: 0.27102167067580385
Brier Score: 0.07764991079507733
xG Sum / Goals Ratio: 0.9979902178929325


In [57]:
plot_importance(xgb_model, max_num_features=10)
plt.show()

In [58]:
sample_with_xg = X_test.copy()
sample_with_xg["xG_pred"] = y_pred_xgb

# 5. Evaluation

In [59]:
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
xG_all = LogRegModel.predict_proba(X)[:, 1]
xGb_all = xgb_model.predict_proba(X)[:, 1]
data_xG = X.copy()
data_xG["goal"] = y
data_xG["xG_LogReg"] =xG_all
data_xG["xG_xGBoost"] = xGb_all

In [60]:
data_xG = data_xG.join(sample[["shot_statsbomb_xg","team","match_id","competition_id","season_id","player"]])
data_xG

,angle,distance,under_pressure,shot_first_time,shot_one_on_one,shot_body_part_Head,shot_body_part_Left Foot,shot_body_part_Other,shot_body_part_Right Foot,shot_technique_Backheel,...,period_4,goal,xG_LogReg,xG_xGBoost,shot_statsbomb_xg,team,match_id,competition_id,season_id,player
0,-0.239906,0.164540,0,1,0,0,0,0,1,0,...,0,0,0.082243,0.074670,0.056644,Werder Bremen,3895302,9,281,Leonardo Bittencourt
1,0.800232,-1.221804,1,1,0,0,1,0,0,0,...,0,0,0.319169,0.267144,0.143381,Bayer Leverkusen,3895302,9,281,Piero Martín Hincapié Reyna
2,-0.703725,0.255942,0,1,0,0,1,0,0,0,...,0,0,0.044723,0.026737,0.038188,Werder Bremen,3895302,9,281,Julián Malatini
3,0.463593,-1.087371,0,0,0,1,0,0,0,0,...,0,0,0.054754,0.091363,0.052781,Bayer Leverkusen,3895302,9,281,Jonathan Tah
4,-0.696439,1.426473,1,0,0,0,1,0,0,0,...,0,0,0.011939,0.011699,0.021272,Bayer Leverkusen,3895302,9,281,Granit Xhaka
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57706,1.190134,-1.203888,1,0,0,1,0,0,0,0,...,0,0,0.102683,0.073572,0.129269,Belgium,7584,43,3,Nacer Chadli
57707,1.778533,-1.367995,0,0,0,1,0,0,0,0,...,0,0,0.163837,0.175380,0.204527,Belgium,7584,43,3,Romelu Lukaku Menama
57708,-0.909751,2.505640,0,0,0,0,0,0,1,0,...,0,0,0.003396,0.005284,0.006300,Belgium,7584,43,3,Thomas Meunier
57709,-0.790763,1.183845,0,0,0,0,1,0,0,0,...,0,0,0.014843,0.006546,0.023417,Belgium,7584,43,3,Jan Vertonghen


In [61]:
data_xG[['goal', 'shot_statsbomb_xg', 'xG_LogReg', 'xG_xGBoost']].sum()

goal                 5927.000000
shot_statsbomb_xg    5671.819937
xG_LogReg            5921.300550
xG_xGBoost           5920.309082
dtype: float64

In [62]:
data_xG.groupby('team')[['goal', 'shot_statsbomb_xg', 'xG_LogReg', 'xG_xGBoost']].sum().sort_values("goal", ascending=False)

,goal,shot_statsbomb_xg,xG_LogReg,xG_xGBoost
team,,,,
Barcelona,895,689.937897,745.410550,745.236938
Paris Saint-Germain,214,175.999358,171.797246,175.615387
Real Madrid,133,103.099928,107.572114,106.731941
Bayer Leverkusen,123,111.570032,115.157734,115.225906
Borussia Dortmund,77,73.294296,66.325503,67.690063
...,...,...,...,...
Córdoba CF,0,0.615977,0.796530,0.624445
Wales,0,1.644895,2.075550,2.046999
Metz,0,0.477597,0.613944,0.553876


In [63]:
data_xG.groupby('player')[['goal', 'shot_statsbomb_xg', 'xG_LogReg', 'xG_xGBoost']].sum().sort_values("goal", ascending=False)

,goal,shot_statsbomb_xg,xG_LogReg,xG_xGBoost
player,,,,
Lionel Andrés Messi Cuccittini,322,218.770414,252.983366,255.660355
Luis Alberto Suárez Díaz,129,101.852185,104.006398,102.504196
Neymar da Silva Santos Junior,74,63.734701,73.532789,72.523575
Kylian Mbappé Lottin,55,39.475857,38.536989,37.686890
Pedro Eliezer Rodríguez Ledesma,45,38.626440,42.586564,41.872559
...,...,...,...,...
Manuel Jesús Arana Rodríguez,0,0.040910,0.039240,0.036322
Manuel Henrique Tavares Fernandes,0,0.047119,0.045128,0.035975
Eberechi Eze,0,0.040455,0.069255,0.048754


In [64]:
WC2022 = data_xG[(data_xG["competition_id"]==43) & (data_xG["season_id"]==106)]
WC2022

,angle,distance,under_pressure,shot_first_time,shot_one_on_one,shot_body_part_Head,shot_body_part_Left Foot,shot_body_part_Other,shot_body_part_Right Foot,shot_technique_Backheel,...,period_4,goal,xG_LogReg,xG_xGBoost,shot_statsbomb_xg,team,match_id,competition_id,season_id,player
55454,-0.427552,0.615941,0,1,0,0,1,0,0,0,...,0,0,0.023419,0.025358,0.036566,Switzerland,3857256,43,106,Granit Xhaka
55455,2.209499,-1.400515,0,1,0,0,1,0,0,0,...,0,0,0.410935,0.410963,0.353289,Switzerland,3857256,43,106,Breel-Donald Embolo
55456,0.121093,-0.294577,0,1,0,0,0,0,1,0,...,0,0,0.073647,0.125234,0.069527,Switzerland,3857256,43,106,Granit Xhaka
55457,1.484520,-1.224110,0,0,0,1,0,0,0,0,...,0,0,0.094376,0.134574,0.081609,Serbia,3857256,43,106,Nikola Milenković
55458,-0.596551,0.730575,0,0,0,0,1,0,0,0,...,0,0,0.033363,0.035750,0.030002,Serbia,3857256,43,106,Andrija Živković
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56831,3.531400,-1.711934,0,0,0,1,0,0,0,0,...,0,0,0.381129,0.370509,0.568312,Denmark,3857254,43,106,Andreas Evald Cornelius
56832,-0.707238,0.562545,0,0,0,0,0,0,1,0,...,0,0,0.036351,0.023262,0.028724,Denmark,3857254,43,106,Mathias Jensen
56833,2.237457,-1.457797,1,0,0,1,0,0,0,0,...,0,0,0.174013,0.143700,0.145394,Denmark,3857254,43,106,Andreas Evald Cornelius
56834,-0.623514,0.948845,0,0,0,0,0,0,1,0,...,0,0,0.027333,0.018752,0.028684,Denmark,3857254,43,106,Joakim Mæhle


In [65]:
WC2022[['goal', 'shot_statsbomb_xg', 'xG_LogReg', 'xG_xGBoost']].sum()

goal                 150.000000
shot_statsbomb_xg    135.884381
xG_LogReg            151.528032
xG_xGBoost           149.683563
dtype: float64

In [66]:
WC2022 = WC2022.join(sample[["x","y"]])
df_goals = WC2022[WC2022["goal"] == 1].copy()
df_non_goal_shots = WC2022[WC2022["goal"] == 0].copy()

# setup the pitch
pitch = VerticalPitch(pad_bottom=0.5,  # pitch extends slightly below halfway line
                      half=True,  # half of a pitch
                      goal_type='box',
                      goal_alpha=0.8, pitch_color='#22312b', line_color='#c7d5cc')  # control the goal transparency

fig, ax = pitch.draw(figsize=(8, 6))

sc1 = pitch.scatter(df_non_goal_shots["x"], df_non_goal_shots["y"],
                    c='#ba4f45',
                    marker='o',
                    ax=ax)
sc2 = pitch.scatter(df_goals["x"], df_goals["y"],
                    c='#ad993c',
                    marker='o',
                    ax=ax)

In [67]:
df_team = WC2022.groupby(["match_id", "team"])[["goal", "shot_statsbomb_xg", "xG_LogReg", "xG_xGBoost"]].sum().reset_index()
df_team["team_number"] = df_team.groupby("match_id").cumcount() + 1
xg_team = df_team.pivot(index="match_id", columns="team_number")
xg_team.columns = [f"{col[0]}{int(col[1])}" for col in xg_team.columns]
xg_team = xg_team.reset_index()
xg_team

,match_id,team1,team2,goal1,goal2,shot_statsbomb_xg1,shot_statsbomb_xg2,xG_LogReg1,xG_LogReg2,xG_xGBoost1,xG_xGBoost2
0,3857254,Denmark,Tunisia,0.0,0.0,1.566559,1.059935,1.470840,1.504847,1.284730,1.202709
1,3857255,Japan,Spain,2.0,1.0,1.157801,0.857712,1.230875,0.879139,0.970948,0.816922
2,3857256,Serbia,Switzerland,2.0,3.0,1.189004,3.103515,1.379405,2.729800,1.443870,2.874526
3,3857257,Australia,Denmark,1.0,0.0,0.469723,0.737155,0.423719,1.241820,0.443730,1.046114
4,3857258,Brazil,Serbia,2.0,0.0,2.060890,0.163327,1.896126,0.161351,1.840314,0.131045
...,...,...,...,...,...,...,...,...,...,...,...
59,3869486,Morocco,Portugal,1.0,0.0,0.972023,0.744121,0.929036,0.871048,0.919463,0.892935
60,3869519,Argentina,Croatia,2.0,0.0,1.080863,0.402532,1.271053,0.684371,1.238454,0.505620
61,3869552,France,Morocco,2.0,0.0,2.002449,1.230364,1.958884,1.248333,2.050494,1.211348
62,3869684,Croatia,Morocco,2.0,1.0,0.849269,0.991457,0.729100,1.193468,0.720507,1.181257


In [68]:
team_form = df_team.groupby("team")[['goal', 'shot_statsbomb_xg', 'xG_LogReg', 'xG_xGBoost']].sum().sort_values("goal", ascending=False)
team_form

,goal,shot_statsbomb_xg,xG_LogReg,xG_xGBoost
team,,,,
France,14,10.181939,10.957163,11.373903
Argentina,11,9.819064,10.546004,10.829145
England,11,6.933062,7.768943,7.684911
Portugal,10,5.690164,6.228713,5.984684
Netherlands,10,4.913616,5.588052,5.382266
Spain,8,3.962028,4.841663,4.444992
Croatia,8,6.894181,7.765704,7.315103
Brazil,7,9.448559,9.982168,10.161661
Morocco,6,5.414087,5.984411,6.254718


In [69]:
team_form["diff"] = team_form["goal"]-team_form["xG_xGBoost"]
team_form.sort_values("diff", ascending=False)

,goal,shot_statsbomb_xg,xG_LogReg,xG_xGBoost,diff
team,,,,,
Netherlands,10,4.913616,5.588052,5.382266,4.617734
Portugal,10,5.690164,6.228713,5.984684,4.015316
Spain,8,3.962028,4.841663,4.444992,3.555008
England,11,6.933062,7.768943,7.684911,3.315089
France,14,10.181939,10.957163,11.373903,2.626097
Ghana,5,2.569753,3.111966,3.132893,1.867107
Serbia,5,3.090398,3.257078,3.311350,1.688650
Australia,3,1.578172,1.534015,1.650714,1.349286
Cameroon,4,3.059733,3.023100,2.826391,1.173609


In [70]:
df_player = WC2022.groupby(["player"])[["goal", "shot_statsbomb_xg", "xG_LogReg", "xG_xGBoost"]].sum().reset_index().sort_values(["goal"], ascending=False)
df_player

,player,goal,shot_statsbomb_xg,xG_LogReg,xG_xGBoost
231,Kylian Mbappé Lottin,6,2.602085,3.529043,3.651713
309,Olivier Giroud,4,3.035955,2.693565,2.728132
208,Julián Álvarez,4,1.908327,2.334444,2.315427
76,Cody Mathès Gakpo,3,0.562367,0.494801,0.600312
67,Bukayo Saka,3,0.579455,1.137063,1.185860
...,...,...,...,...,...
161,Ismaïla Sarr,0,1.065290,1.346351,1.258945
160,Ismail Jakobs,0,0.014393,0.009342,0.011000
159,Ismaeel Mohammad Mohammad,0,0.357907,0.448547,0.371280
158,In-Beom Hwang,0,0.197748,0.178622,0.137516


In [71]:
df_player["diff"] = df_player["goal"]-df_player["xG_xGBoost"]
df_player.sort_values("diff", ascending=False)

,player,goal,shot_statsbomb_xg,xG_LogReg,xG_xGBoost,diff
76,Cody Mathès Gakpo,3,0.562367,0.494801,0.600312,2.399688
231,Kylian Mbappé Lottin,6,2.602085,3.529043,3.651713,2.348287
67,Bukayo Saka,3,0.579455,1.137063,1.185860,1.814140
421,Álvaro Borja Morata Martín,3,1.109481,1.358027,1.232216,1.767784
327,Rafael Alexandre Conceição Leão,2,0.248384,0.296429,0.302849,1.697151
...,...,...,...,...,...,...
187,Jonathan David,0,0.812814,1.139006,1.210857,-1.210857
161,Ismaïla Sarr,0,1.065290,1.346351,1.258945,-1.258945
171,Jamal Musiala,0,1.152516,1.358513,1.495016,-1.495016
348,Romelu Lukaku Menama,0,1.684155,1.600732,1.553446,-1.553446


In [72]:
mbappe = WC2022[WC2022["player"].str.contains('Kylian')]
mbappe_goals = mbappe[mbappe["goal"] == 1].copy()
mbappe_non_goal_shots = mbappe[mbappe["goal"] == 0].copy()

pitch = VerticalPitch(pad_bottom=0.5, half=True, goal_type='box', goal_alpha=0.8, pitch_color='#22312b', line_color='#c7d5cc')
fig, ax = pitch.draw(figsize=(8, 6))
ax.text(80, 123, 'Kylian Mbappé Shot Map', color='white', fontsize=14, ha='right', va='top', fontweight='bold')
sc1 = pitch.scatter(mbappe_non_goal_shots["x"], mbappe_non_goal_shots["y"], c='#ba4f45', marker='o', ax=ax)
sc2 = pitch.scatter(mbappe_goals["x"], mbappe_goals["y"], c='#ad993c', marker='o', ax=ax)

# 6. Deployment

In [73]:
import joblib
joblib.dump(xgb_model, "xg_model.pkl")

['xg_model.pkl']

In [75]:
%%writefile xg_app.py
import streamlit as st
import pandas as pd
import joblib

# Load trained model
model = joblib.load("xg_model.pkl")

st.title("⚽ Expected Goals (xG) Predictor")

st.markdown("Enter shot features below to estimate the xG value:")

# Numeric inputs
distance = st.number_input("Shot Distance (meters)", min_value=0.0, max_value=100.0, value=10.0)
angle = st.number_input("Shot Angle (degrees)", min_value=0.0, max_value=180.0, value=45.0)

# Binary inputs (checkboxes)
under_pressure = st.checkbox("Under Pressure?", value=False)
shot_first_time = st.checkbox("First Time Shot?", value=False)
shot_one_on_one = st.checkbox("One-on-One with GK?", value=False)

# Body part (one-hot encoded)
body_part = st.selectbox("Body Part", ["Head", "Left Foot", "Right Foot", "Other"])

# Technique (one-hot encoded)
technique = st.selectbox( "Shot Technique", 
                         ["Normal", "Backheel", "Diving Header", "Half Volley", "Lob","Overhead Kick", "Volley"])

# Position (one-hot encoded)
position = st.selectbox(
    "Player Position",
    [
        "Goalkeeper", "Center Back", "Left Back", "Right Back",
        "Left Wing Back", "Right Wing Back",
        "Left Defensive Midfield", "Right Defensive Midfield",
        "Center Defensive Midfield",
        "Left Midfield", "Center Midfield", "Right Midfield",
        "Left Attacking Midfield", "Center Attacking Midfield", "Right Attacking Midfield",
        "Left Center Forward", "Right Center Forward", "Secondary Striker"
    ]
)

# Period (first half, second half, etc.)
period = st.selectbox("Match Period", ["1", "2", "3", "4"])

# Initialize all features to 0
features = {
    'angle': angle,
    'distance': distance,
    'under_pressure': int(under_pressure),
    'shot_first_time': int(shot_first_time),
    'shot_one_on_one': int(shot_one_on_one)
}

# Add one-hot encoded features
# Body parts
for bp in ['Head', 'Left Foot', 'Right Foot', 'Other']:
    features[f'shot_body_part_{bp}'] = 1 if body_part == bp else 0

# Techniques
for t in ['Backheel', 'Diving Header', 'Half Volley', 'Lob', 'Normal', 'Overhead Kick', 'Volley']:
    features[f'shot_technique_{t}'] = 1 if technique == t else 0

# Positions
positions = [
    "Center Attacking Midfield", "Center Back", "Center Defensive Midfield", "Center Forward", "Center Midfield",
    "Goalkeeper", "Left Attacking Midfield", "Left Back", "Left Center Forward", "Left Center Midfield",
    "Left Defensive Midfield", "Left Midfield", "Left Wing", "Left Wing Back",
    "Right Attacking Midfield", "Right Back", "Right Center Back", "Right Center Forward", "Right Center Midfield",
    "Right Defensive Midfield", "Right Midfield", "Right Wing", "Right Wing Back",
    "Secondary Striker"
]
for pos in positions:
    features[f'position_{pos}'] = 1 if position == pos else 0

# Periods
for p in ['1', '2', '3', '4']:
    features[f'period_{p}'] = 1 if period == p else 0

# Make prediction
if st.button("Predict xG"):
    input_df = pd.DataFrame([features])
    # Ensure all expected columns exist (fill any missing with 0)
    expected_features = model.get_booster().feature_names
    for col in expected_features:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[expected_features]

    # Predict
    xg_pred = model.predict_proba(input_df)[:, 1][0]
    st.success(f"Estimated xG: **{xg_pred:.3f}**")

Writing xg_app.py


In [ ]:
#cd C:\Users\FILIP\Desktop\projekti\xG model
#streamlit run xg_app.py